# Camada SILVER — Limpeza e Enriquecimento
Lê da Bronze, corrige tipos, remove nulos e junta as tabelas.

In [15]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

spark = get_spark('Silver - Transformação')
spark.sparkContext.setLogLevel('WARN')

BRONZE = '/opt/spark/work-dir/warehouse/bronze'
SILVER = '/opt/spark/work-dir/warehouse/silver'

26/06/18 23:25:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [16]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

clientes = (
    spark.read.format('delta').load(f'{BRONZE}/clientes')
    .withColumn(
        'documento',
        F.regexp_replace(F.col('documento'), r'[^0-9]', '')
    )
    .withColumn('data_cadastro', F.col('data_cadastro').cast(DateType()))
    .withColumn('ativo', F.col('ativo').cast('boolean'))
    .filter(F.col('cliente_id').isNotNull())
)

clientes.show(5)
print('Clientes:', clientes.count())

+----------+--------------------+------------+--------------+------------+-------------+-------------+-----+-------------+
|cliente_id|                nome|tipo_cliente|     documento|      regiao|       cidade|data_cadastro|ativo|score_credito|
+----------+--------------------+------------+--------------+------------+-------------+-------------+-----+-------------+
|         1|  Maria Luísa Borges|          PF|   31705892450|     Sudeste|       Santos|   2022-01-01| true|        642.0|
|         2|                Leão|          PJ|79384160000131|         Sul|Florianópolis|   2022-09-03| true|        914.6|
|         3|        Sophia Souza|          PF|   21463970803|         Sul|Caxias do Sul|   2024-04-15| true|        832.4|
|         4|Sra. Rebeca Cassiano|          PF|   78926431509|     Sudeste|     Campinas|   2023-07-17| true|        669.4|
|         5|     Isaque Oliveira|          PF|   63902841796|Centro-Oeste|      Goiânia|   2023-07-07| true|        782.9|
+----------+----

In [17]:
# Produtos — adiciona coluna 'disponivel'
produtos = (
    spark.read.format('delta').load(f'{BRONZE}/produtos')
    .withColumn('preco',     F.col('preco').cast('double'))
    .withColumn('estoque',   F.col('estoque').cast('integer'))
    .withColumn('disponivel', F.col('estoque') > 0)
)
produtos.show(5)
print('Produtos:', produtos.count())

+----------+-----------------+---------+------+-------+-------+-------------+----------+
|produto_id|             nome|categoria| preco|estoque|peso_kg|fornecedor_id|disponivel|
+----------+-----------------+---------+------+-------+-------+-------------+----------+
|         1|       Lençol 481|     Casa| 91.76|    319|   4.29|           28|      true|
|         2|        Tênis 249| Esportes|780.55|    256|   5.46|           48|      true|
|         3|Condicionador 963|   Beleza|107.04|    414|  17.55|           20|      true|
|         4|   Esfoliante 132|   Beleza|114.86|     28|  27.63|           32|      true|
|         5|   Ventilador 789|     Casa|273.16|    302|  14.79|           29|      true|
+----------+-----------------+---------+------+-------+-------+-------------+----------+
only showing top 5 rows
Produtos: 1000


In [4]:
# Vendas — filtra CONCLUIDO, calcula valor_total e junta dimensões
vendas = (
    spark.read.format('delta').load(f'{BRONZE}/vendas')
    .withColumn('data_venda',   F.col('data_venda').cast(DateType()))
    .withColumn('quantidade',   F.col('quantidade').cast('integer'))
    .withColumn('desconto_pct', F.col('desconto_pct').cast('double'))
    .filter(F.col('status') == 'CONCLUIDO')
    .join(produtos.select('produto_id', 'preco', 'nome', 'categoria'), 'produto_id')
    .join(clientes.select('cliente_id', 'regiao'), 'cliente_id')
    .withColumn(
        'valor_total',
        F.round(F.col('preco') * F.col('quantidade') * (1 - F.col('desconto_pct') / 100), 2)
    )
    .withColumn('ano_mes', F.date_format('data_venda', 'yyyy-MM'))
)
vendas.select('venda_id', 'nome', 'categoria', 'regiao', 'quantidade', 'valor_total', 'ano_mes').show(10)
print('Vendas CONCLUIDAS:', vendas.count())

+--------+-----------------+----------+------------+----------+-----------+-------+
|venda_id|             nome| categoria|      regiao|quantidade|valor_total|ano_mes|
+--------+-----------------+----------+------------+----------+-----------+-------+
|       1|     Película 493|Automotivo|     Sudeste|         7|   12495.04|2022-07|
|       4|Princípios de 525|    Livros|Centro-Oeste|         8|   14275.01|2022-02|
|       5|      Arte de 268|    Livros|    Nordeste|         8|   25152.48|2024-01|
|       6|     Camiseta 965|    Roupas|Centro-Oeste|         5|   22321.65|2023-09|
|       7|      Perfume 547|    Beleza|     Sudeste|        10|    2728.75|2022-09|
|       8|      Jaqueta 855|    Roupas|     Sudeste|         5|      469.7|2022-01|
|      10|         Capa 562|Automotivo|       Norte|         6|   18813.71|2023-09|
|      11|     Camiseta 965|    Roupas|     Sudeste|         4|   17857.32|2023-07|
|      14|  Tapete Auto 352|Automotivo|       Norte|         9|   16871.69|2

Vendas CONCLUIDAS: 599930


In [5]:
# Persiste na Silver
for name, df in [('clientes', clientes), ('produtos', produtos), ('vendas', vendas)]:
    df.write.format('delta').mode('overwrite').save(f'{SILVER}/{name}')
    print(f'[silver/{name}] salvo — {df.count()} linhas')

[silver/clientes] salvo — 100000 linhas


[silver/produtos] salvo — 1000 linhas


[silver/vendas] salvo — 599930 linhas
